

**Steps:**

1. Use the risk-neutral dynamics from eq. (2.2):
$ dS_t = r\, S_t\, dt + \mathbf{B}(S_t)\, dW_t $. NO DRIFT except for the risk-free rate $r$.

2. Represent the volatility matrix $B$ as
   $B = (B^i_{j,k})_{i,j,k\in\{1,\dots,n\}}$. In 2D that is *two
   full $2\times2$ matrices* $B^1, B^2$.

3. Use equation (4.8): $\tilde S^{(L)}_{t_{j+1}} = \tilde S^{(L)}_{t_j}\big[1 + r(t_{j+1} - t_j)\big] + \mathbf{B}\big(\tilde S^{(L)}_{t_j}\big)\sqrt{t_{j+1} - t_j}\, G_j$

4. Monte Carlo over $M$ paths to evaluate $e^{-rT}\,\mathbb{E}[\Phi(S_\cdot)]$.

5. Invert Black–Scholes on the resulting call prices to get the implied-vol
   surface.



In [53]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from scipy.optimize import brentq

np.set_printoptions(precision=4, suppress=True)


In [54]:
import plotly.io as pio
pio.renderers.default = 'notebook_connected'


**The EBS volatility operator**

The matrix $B$ is encoded as a two-by-two matrix using B[i, j, k] with $i, j, k \in \{0, 1\}$, where index $i$ runs over Brownian components and $(j, k)$ over assets. Following eq. (2.6), $ dS^k_t = r\, S^k_t\, dt + \sum_{i,j=1}^{n} \mathbf{B}^i_{j,k}\, S^j_t\, dW^i_t. $

Equivalently, for each Brownian component $i$ we have a matrix $\mathbf{B}^i$ acting
on the price vector. The diffusion coefficient of the $k$-th asset against $dW^i$ is $\sum_j \mathbf{B}^i_{j,k}\, S^j_t$.

**Note:** The paper prints two two-by-two matrices $\mathbf{B}^1, \mathbf{B}^2$. Index was previosuly incorrect. The format [j,k] does not work because (price-multiplier row, diffused-asset column) and plug $S_0 =(103.8, 26.11)$ into eq. (2.7), the spot IBM volatility comes out at 130% and DELL at 2400%. Reading them as [k, j] (diffused-asset row, price-multiplier column) gives IBM ≈ 39%, DELL ≈ 58%, which matches the figures. Therefore, the second method transposes the matrices into the [i,j,k] format.


In [55]:
B1_printed = np.array([[ 0.2445,  0.2774],
                       [ 0.4155, -1.3032]])
B2_printed = np.array([[-1.3611,  6.2997],
                       [ 0.4044, -1.1423]])

B_tensor = np.stack([B1_printed.T, B2_printed.T], axis=0)   # B_tensor[i, j, k]
print("B_tensor shape:", B_tensor.shape)
print("B^1 (as [j, k]) =\n", B_tensor[0])
print("B^2 (as [j, k]) =\n", B_tensor[1])


B_tensor shape: (2, 2, 2)
B^1 (as [j, k]) =
 [[ 0.2445  0.4155]
 [ 0.2774 -1.3032]]
B^2 (as [j, k]) =
 [[-1.3611  0.4044]
 [ 6.2997 -1.1423]]


In [56]:
#Check spot volatility
S0_check = np.array([103.8, 26.11])   # IBM, DELL spot 02/28/02
for k in range(2):
    inside = sum(sum(B_tensor[i, j, k] * S0_check[j] for j in range(2))**2 for i in range(2))
    sigma_k = np.sqrt(inside) / S0_check[k]
    name = "IBM" if k == 0 else "DELL"
    print(f"{name} instantaneous vol at spot: {sigma_k*100:.2f}%")


IBM instantaneous vol at spot: 38.57%
DELL instantaneous vol at spot: 58.15%


Equation (4.8):
$ \tilde S_{t_{j+1}} = \tilde S_{t_j}(1 + r\,\Delta t)+ \sum_{i=1}^{n} \mathbf{B}^i(\tilde S_{t_j})\,\sqrt{\Delta t}\, G^i_j $ where $\mathbf{B}^i(S)$ is the vector with $k$-th entry $\sum_j \mathbf{B}^i_{j,k} S^j$.
Note this is computed from the tensor (transpose because $k$ is the column index in our convention and we want a vector indexed by $k$). The function below vectorizes over `M` Monte Carlo paths simultaneously.


In [57]:
def simulate_paths(S0, B_tensor, r, T, L, M, seed=None):
    """
    Parameters:
    S0 : prices
    B_tensor : (n, n, n) array     volatility operator B[i, j, k]
    r : risk-free rate
    T : time
    L : number of time steps
    M : number of Monte Carlo paths
    seed : seed
    """
    rng = np.random.default_rng(seed)
    n = S0.shape[0]
    dt = T / L
    sqrt_dt = np.sqrt(dt)

    S = np.empty((M, L + 1, n))
    S[:, 0, :] = S0

    G = rng.standard_normal(size=(M, L, n))  # G[m, j, i]

    for j in range(L):
        s = S[:, j, :]                      # shape (M, n)
        drift = r * s * dt                  # # Drift: r * S * dt = (M, n)
        BS = np.einsum('mj,ijk->mik', s, B_tensor)   # (M, n_i, n_k)
        diffusion = np.einsum('mik,mi->mk', BS, G[:, j, :]) * sqrt_dt
        S[:, j + 1, :] = s + drift + diffusion

    return S


**Check**
Before pricing anything, verify that the local
covariance are similair to the given volatily matrices. For a small $\Delta t$,

$ \mathrm{Cov}(\Delta S \mid S_t) \approx \Delta t \cdot \mathbf{B}(S_t)\mathbf{B}(S_t)^T $

where the $(k, l)$ entry of $\mathbf{B}(S)\mathbf{B}(S)^T$ is
$\sum_i \big(\sum_j \mathbf{B}^i_{j,k} S^j\big)\big(\sum_j \mathbf{B}^i_{j,l} S^j\big)$.


In [63]:
S0 = np.array([103.8, 26.11])   # IBM, DELL spot on 02/28/02 per the paper
r = 0.0179
T_small = 1.0 / 252
L_small = 1
M_check = 200_000

S_check = simulate_paths(S0, B_tensor, r, T_small, L_small, M_check, seed=0)
dS = S_check[:, 1, :] - S_check[:, 0, :]

empirical_cov = np.cov(dS.T)

# Theoretical local covariance from B(S0) B(S0)^T times dt
BS0 = np.einsum('j,ijk->ik', S0, B_tensor)     # (n_i, n_k) — B^i(S0) as rows
theoretical_cov = BS0.T @ BS0 * T_small

print("Empirical covariance of ΔS over dt:")
print(empirical_cov)
print()
print("Theoretical B(S0) B(S0)^T * dt:")
print(theoretical_cov)
print()
print("Ratio (should be ≈ 1):")
print(empirical_cov / theoretical_cov)


Empirical covariance of ΔS over dt:
[[6.3564 2.2948]
 [2.2948 0.9139]]

Theoretical B(S0) B(S0)^T * dt:
[[6.3594 2.2972]
 [2.2972 0.9147]]

Ratio (should be ≈ 1):
[[0.9995 0.999 ]
 [0.999  0.9991]]


Equation 4.1: 

$ \mathrm{Call}(K, T) = e^{-rT}\, \mathbb{E}\big[(S^i_T - K)^+\big] $

Price on a (strike × maturity) grid for IBM (asset index 0). The paper uses $L=100$ discretization steps and $M=10^6$ paths.

In [64]:
def price_vanilla_calls(S0, B_tensor, r, strikes, maturities, asset_idx,
                         L=100, M=1000000, seed=1):
    """
    Returns a (len(maturities), len(strikes)) array of call prices on a single
    underlying, computed by Monte Carlo.
    """
    prices = np.zeros((len(maturities), len(strikes)))

    for i, T in enumerate(maturities):
        S = simulate_paths(S0, B_tensor, r, T, L, M, seed=seed + i)
        ST = S[:, -1, asset_idx]                          # terminal price of chosen asset
        disc = np.exp(-r * T)
        payoffs = np.maximum(ST[:, None] - strikes[None, :], 0.0)
        prices[i, :] = disc * payoffs.mean(axis=0)

    return prices


# Grid: strikes as % of spot, maturities in days.
strike_pcts = np.array([60, 70, 80, 90, 100, 110, 120, 130, 140, 150]) / 100.0
maturities_days = np.array([30, 60, 90, 120, 150, 180, 360, 720])
maturities = maturities_days / 365.0
strikes = strike_pcts * S0[0]        # strikes on IBM, in dollars

call_prices = price_vanilla_calls(S0, B_tensor, r, strikes, maturities,
                                   asset_idx=0, L=100, M=100_000, seed=42)

print("Call prices (% of spot) on IBM grid:")
print("Rows = maturities (days), Cols = strikes (% of spot)")
print("          ", "  ".join(f"{p*100:>5.0f}%" for p in strike_pcts))
for i, d in enumerate(maturities_days):
    row = call_prices[i] / S0[0] * 100
    print(f"  T={d:>4}d  ", "  ".join(f"{v:>5.2f}%" for v in row))


Call prices (% of spot) on IBM grid:
Rows = maturities (days), Cols = strikes (% of spot)
              60%     70%     80%     90%    100%    110%    120%    130%    140%    150%
  T=  30d   40.08%  30.09%  20.16%  11.02%   4.58%   1.46%   0.37%   0.08%   0.01%   0.00%
  T=  60d   40.23%  30.31%  20.78%  12.61%   6.76%   3.25%   1.43%   0.59%   0.24%   0.09%
  T=  90d   40.39%  30.66%  21.65%  14.15%   8.62%   4.96%   2.74%   1.49%   0.80%   0.44%
  T= 120d   40.64%  31.20%  22.67%  15.62%  10.29%   6.53%   4.04%   2.49%   1.54%   0.98%
  T= 150d   40.87%  31.74%  23.61%  16.91%  11.72%   7.93%   5.29%   3.53%   2.37%   1.62%
  T= 180d   41.37%  32.51%  24.71%  18.25%  13.18%   9.37%   6.60%   4.65%   3.31%   2.38%
  T= 360d   44.40%  36.81%  30.20%  24.62%  20.00%  16.24%  13.22%  10.80%   8.89%   7.37%
  T= 720d   58.06%  51.69%  46.10%  41.23%  36.99%  33.32%  30.14%  27.41%  25.05%  23.02%


In [60]:
import plotly.graph_objects as go

K_pct_grid, T_grid = np.meshgrid(strike_pcts * 100, maturities_days)
Z_calls = call_prices / S0[0] * 100   # call price in % of spot

fig = go.Figure(data=[go.Surface(
    x=K_pct_grid, y=T_grid, z=Z_calls,
    colorscale=[[0, 'white'], [1, 'black']],
    showscale=False,
    contours=dict(
        x=dict(show=True, color='black', width=1),
        y=dict(show=True, color='black', width=1),
    ),
    lighting=dict(ambient=0.7, diffuse=0.3, specular=0.05),
    hidesurface=False,
)])
fig.update_layout(
    title='Model call option prices on IBM after parameter estimation',
    scene=dict(
        xaxis_title='Strike price in % of the spot',
        yaxis_title='Time to maturity in days',
        zaxis_title='Call price in % of the spot',
        xaxis=dict(backgroundcolor='white', gridcolor='lightgrey', showbackground=True),
        yaxis=dict(backgroundcolor='white', gridcolor='lightgrey', showbackground=True),
        zaxis=dict(backgroundcolor='white', gridcolor='lightgrey', showbackground=True),
    ),
    paper_bgcolor='white',
    plot_bgcolor='white',
    width=750, height=550,
    margin=dict(l=0, r=0, t=40, b=0),
)
fig.show()


In [61]:
#implied volatiliy surface
def bs_call_price(S, K, T, r, sigma):
    if T <= 0 or sigma <= 0:
        return max(S - K * np.exp(-r * T), 0.0)
    d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)


def implied_vol(price, S, K, T, r):
    """Brent-method implied vol. Returns NaN if out of arbitrage bounds."""
    intrinsic = max(S - K * np.exp(-r * T), 0.0)
    upper = S
    if price < intrinsic - 1e-8 or price > upper:
        return np.nan
    f = lambda s: bs_call_price(S, K, T, r, s) - price
    try:
        return brentq(f, 1e-6, 5.0, xtol=1e-6, maxiter=200)
    except ValueError:
        return np.nan


iv_surface = np.full_like(call_prices, np.nan)
for i, T in enumerate(maturities):
    for k, K in enumerate(strikes):
        iv_surface[i, k] = implied_vol(call_prices[i, k], S0[0], K, T, r)

print("Implied vol surface (%):")
print("          ", "  ".join(f"{p*100:>5.0f}%" for p in strike_pcts))
for i, d in enumerate(maturities_days):
    row = iv_surface[i] * 100
    print(f"  T={d:>4}d  ", "  ".join(f"{v:>5.2f}" if np.isfinite(v) else "  nan" for v in row))


Implied vol surface (%):
              60%     70%     80%     90%    100%    110%    120%    130%    140%    150%
  T=  30d     nan    nan  35.84  37.94  39.46  40.71  41.63  42.29  42.17  41.71
  T=  60d   54.31  43.06  40.12  40.26  40.96  41.67  42.30  42.90  43.52  44.15
  T=  90d   49.41  43.47  42.04  42.10  42.56  43.12  43.69  44.37  45.12  46.04
  T= 120d   49.34  45.15  43.91  43.72  43.91  44.21  44.62  45.18  45.91  46.77
  T= 150d   47.70  45.52  44.70  44.61  44.69  44.94  45.30  45.81  46.45  47.26
  T= 180d   50.30  47.45  46.29  45.89  45.85  46.00  46.29  46.69  47.25  47.93
  T= 360d   53.44  51.30  50.08  49.45  49.14  49.06  49.15  49.36  49.66  50.05
  T= 720d   80.01  74.38  70.62  68.06  66.32  65.11  64.30  63.80  63.53  63.44


In [62]:
fig = go.Figure(data=[go.Surface(
    x=K_pct_grid, y=T_grid, z=iv_surface * 100,
    colorscale=[[0, 'white'], [1, 'black']],
    showscale=False,
    contours=dict(
        x=dict(show=True, color='black', width=1),
        y=dict(show=True, color='black', width=1),
    ),
    lighting=dict(ambient=0.7, diffuse=0.3, specular=0.05),
)])
fig.update_layout(
    title='Model implied volatility for IBM after parameter estimation',
    scene=dict(
        xaxis_title='Strike price in % of the spot',
        yaxis_title='Time to maturity in days',
        zaxis_title='Volatility in %',
        xaxis=dict(backgroundcolor='white', gridcolor='lightgrey', showbackground=True),
        yaxis=dict(backgroundcolor='white', gridcolor='lightgrey', showbackground=True),
        zaxis=dict(backgroundcolor='white', gridcolor='lightgrey', showbackground=True),
    ),
    paper_bgcolor='white',
    plot_bgcolor='white',
    width=750, height=550,
    margin=dict(l=0, r=0, t=40, b=0),
)
fig.show()
